In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
DATA_PATH = ROOT / "data" / "processed" / "ml_dataset.csv"

df = pd.read_csv(DATA_PATH)

df["trade_date"] = pd.to_datetime(df["trade_date"])

print(df.shape)
display(df.head())

(2685, 28)


,stock_code,trade_date,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,...,macd,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20,target_return_5d
0,660,2024-06-11,0.021635,0.094233,0.054591,0.181212,0.011905,0.043689,0.009615,203000.0,...,6187.902397,5195.016563,992.885834,0.031048,0.023996,6757.468051,0.253659,3435519.80,0.893653,0.103529
1,660,2024-06-12,0.011765,0.112261,0.061728,0.169750,0.014151,0.023697,-0.002353,207340.0,...,6911.664995,5538.346249,1373.318746,0.028770,0.023814,6631.934619,-0.299278,3381585.20,0.636190,0.086047
2,660,2024-06-13,0.032558,0.146102,0.096296,0.198057,-0.017699,0.034247,0.051163,213000.0,...,7958.354609,6022.347921,1936.006687,0.026692,0.024432,6979.653575,1.685444,3532295.70,1.635559,0.069820
3,660,2024-06-14,-0.004505,0.065060,0.129280,0.145078,-0.017778,0.041667,0.013514,215700.0,...,8607.944994,6539.467336,2068.477659,0.014806,0.023386,7123.964034,-0.426854,3443697.95,0.961531,0.058824
4,660,2024-06-17,0.009050,0.072115,0.178647,0.174302,0.018265,0.047945,-0.009050,218700.0,...,9178.331251,7067.240119,2111.091132,0.013915,0.022745,7365.109460,-0.336094,3411545.25,0.644382,0.000000


In [3]:
TARGET = "target_return_5d"

EXCLUDE_COLUMNS = [
    "stock_code",
    "trade_date",
    TARGET
]

FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in EXCLUDE_COLUMNS
]

print("Feature count:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)

Feature count: 25
['return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20']


In [4]:
train = df[
    (df["trade_date"] >= "2024-03-13") &
    (df["trade_date"] <= "2025-12-31")
].copy()

validation = df[
    (df["trade_date"] >= "2026-01-01") &
    (df["trade_date"] <= "2026-06-30")
].copy()

test = df[
    (df["trade_date"] >= "2026-07-01") &
    (df["trade_date"] <= "2026-09-01")
].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (1895, 28)
Validation: (600, 28)
Test: (190, 28)


In [5]:
def create_xgb():
    return XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror",
        n_jobs=-1
    )

In [6]:
X_train = train[FEATURE_COLUMNS]
y_train = train[TARGET]

X_val = validation[FEATURE_COLUMNS]
y_val = validation[TARGET]

X_test = test[FEATURE_COLUMNS]
y_test = test[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (1895, 25)
y_train: (1895,)


In [7]:
feature_counts = [5, 10, 15, 20]
tol_values = [0, 0.001, 0.005]

cv = KFold(
    n_splits=5,
    shuffle=False
)

results = []

for n_features in feature_counts:
    for tol in tol_values:

        print("=" * 70)
        print(
            f"SFS XGBoost | "
            f"n_features={n_features}, tol={tol}"
        )

        estimator = create_xgb()

        sfs = SequentialFeatureSelector(
            estimator=estimator,
            n_features_to_select=n_features,
            direction="forward",
            tol=tol,
            scoring="neg_mean_squared_error",
            cv=cv,
            n_jobs=-1
        )

        sfs.fit(X_train, y_train)

        selected_features = X_train.columns[
            sfs.get_support()
        ].tolist()

        n_selected = len(selected_features)

        print("Selected features:", n_selected)
        print(selected_features)

        # Validation 평가
        final_model = create_xgb()

        final_model.fit(
            X_train[selected_features],
            y_train
        )

        val_pred = final_model.predict(
            X_val[selected_features]
        )

        mse = mean_squared_error(y_val, val_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_val, val_pred)
        r2 = r2_score(y_val, val_pred)

        results.append({
            "n_features_to_select": n_features,
            "tol": tol,
            "n_selected_features": n_selected,
            "selected_features": selected_features,
            "MSE": mse,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        })

        print(
            f"MSE={mse:.6f}, "
            f"RMSE={rmse:.6f}, "
            f"MAE={mae:.6f}, "
            f"R2={r2:.6f}"
        )

SFS XGBoost | n_features=5, tol=0
Selected features: 5
['return_1d', 'return_5d', 'high_low_range', 'price_to_sma_20', 'rsi_14']
MSE=0.012397, RMSE=0.111340, MAE=0.083110, R2=-0.131235
SFS XGBoost | n_features=5, tol=0.001
Selected features: 5
['return_1d', 'return_5d', 'high_low_range', 'price_to_sma_20', 'rsi_14']
MSE=0.012397, RMSE=0.111340, MAE=0.083110, R2=-0.131235
SFS XGBoost | n_features=5, tol=0.005
Selected features: 5
['return_1d', 'return_5d', 'high_low_range', 'price_to_sma_20', 'rsi_14']
MSE=0.012397, RMSE=0.111340, MAE=0.083110, R2=-0.131235
SFS XGBoost | n_features=10, tol=0
Selected features: 10
['return_1d', 'return_5d', 'intraday_return', 'high_low_range', 'gap', 'price_to_sma_5', 'price_to_sma_20', 'rsi_14', 'macd_signal', 'volume_change_1d']
MSE=0.013272, RMSE=0.115203, MAE=0.085980, R2=-0.211104
SFS XGBoost | n_features=10, tol=0.001
Selected features: 10
['return_1d', 'return_5d', 'intraday_return', 'high_low_range', 'gap', 'price_to_sma_5', 'price_to_sma_20', 'r

In [8]:
sfs_xgb_results = pd.DataFrame(results)

sfs_xgb_results

,n_features_to_select,tol,n_selected_features,selected_features,MSE,RMSE,MAE,R2
0,5,0.000,5,"[return_1d, return_5d, high_low_range, price_t...",0.012397,0.111340,0.083110,-0.131235
1,5,0.001,5,"[return_1d, return_5d, high_low_range, price_t...",0.012397,0.111340,0.083110,-0.131235
2,5,0.005,5,"[return_1d, return_5d, high_low_range, price_t...",0.012397,0.111340,0.083110,-0.131235
3,10,0.000,10,"[return_1d, return_5d, intraday_return, high_l...",0.013272,0.115203,0.085980,-0.211104
4,10,0.001,10,"[return_1d, return_5d, intraday_return, high_l...",0.013272,0.115203,0.085980,-0.211104
5,10,0.005,10,"[return_1d, return_5d, intraday_return, high_l...",0.013272,0.115203,0.085980,-0.211104
6,15,0.000,15,"[return_1d, return_5d, return_10d, intraday_re...",0.013052,0.114247,0.085335,-0.191080
7,15,0.001,15,"[return_1d, return_5d, return_10d, intraday_re...",0.013052,0.114247,0.085335,-0.191080
8,15,0.005,15,"[return_1d, return_5d, return_10d, intraday_re...",0.013052,0.114247,0.085335,-0.191080
9,20,0.000,20,"[return_1d, return_5d, return_10d, return_20d,...",0.012397,0.111340,0.084208,-0.131242


In [9]:
sfs_xgb_results_sorted = (
    sfs_xgb_results
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(sfs_xgb_results_sorted)

,n_features_to_select,tol,n_selected_features,selected_features,MSE,RMSE,MAE,R2
0,5,0.000,5,"[return_1d, return_5d, high_low_range, price_t...",0.012397,0.111340,0.083110,-0.131235
1,5,0.001,5,"[return_1d, return_5d, high_low_range, price_t...",0.012397,0.111340,0.083110,-0.131235
2,5,0.005,5,"[return_1d, return_5d, high_low_range, price_t...",0.012397,0.111340,0.083110,-0.131235
3,20,0.000,20,"[return_1d, return_5d, return_10d, return_20d,...",0.012397,0.111340,0.084208,-0.131242
4,20,0.001,20,"[return_1d, return_5d, return_10d, return_20d,...",0.012397,0.111340,0.084208,-0.131242
5,20,0.005,20,"[return_1d, return_5d, return_10d, return_20d,...",0.012397,0.111340,0.084208,-0.131242
6,15,0.000,15,"[return_1d, return_5d, return_10d, intraday_re...",0.013052,0.114247,0.085335,-0.191080
7,15,0.001,15,"[return_1d, return_5d, return_10d, intraday_re...",0.013052,0.114247,0.085335,-0.191080
8,15,0.005,15,"[return_1d, return_5d, return_10d, intraday_re...",0.013052,0.114247,0.085335,-0.191080
9,10,0.000,10,"[return_1d, return_5d, intraday_return, high_l...",0.013272,0.115203,0.085980,-0.211104


In [10]:
OUTPUT_DIR = ROOT / "data" / "processed" / "filter_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_PATH = OUTPUT_DIR / "sfs_xgboost_results.csv"

sfs_xgb_results.to_csv(
    RESULT_PATH,
    index=False
)

print(f"Saved: {RESULT_PATH}")

Saved: /Users/yangjaehoon/Desktop/StockLens/data/processed/filter_results/sfs_xgboost_results.csv


In [11]:
best_result = sfs_xgb_results_sorted.iloc[0]

print("Best SFS + XGBoost configuration")
print("-" * 50)

print(
    "n_features_to_select:",
    best_result["n_features_to_select"]
)

print(
    "tol:",
    best_result["tol"]
)

print(
    "n_selected_features:",
    best_result["n_selected_features"]
)

print(
    "Selected features:",
    best_result["selected_features"]
)

print(
    "Validation RMSE:",
    best_result["RMSE"]
)

print(
    "Validation MAE:",
    best_result["MAE"]
)

print(
    "Validation R2:",
    best_result["R2"]
)

Best SFS + XGBoost configuration
--------------------------------------------------
n_features_to_select: 5
tol: 0.0
n_selected_features: 5
Selected features: ['return_1d', 'return_5d', 'high_low_range', 'price_to_sma_20', 'rsi_14']
Validation RMSE: 0.11133995809178453
Validation MAE: 0.08311011620185517
Validation R2: -0.1312350421866033
